In [1]:
# loader_csi_dataset.py
import os
import re
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import numpy as np
import pandas as pd


# ======== Cấu hình lưới ========
# Geometry main:
#   - TX tại (0, 0.35, 0.3)
#   - Scan0 / Scan1: 10x6 grid
#   - Ref: 2x4 grid
#
# Geometry opposite:
#   - layout opposite dùng TX_OP / RX_OP
#   - folder opposite có thể nhiều tầng
#   - point_id phải được suy từ numeric ancestor gần file nhất
#     để tránh collapse toàn bộ packet về top-level dir

SCAN_NCOLS = 6   # cột theo chiều y
SCAN_NROWS = 10  # hàng theo chiều z

Z_ROWS = 2
Z_COLS = 4

FREQ_WHITELIST = {5180, 5200, 5220, 5240, 5280}
SESSION_RX_RE = re.compile(r"rx_\d+_(\d{6})_(\d{6})", re.IGNORECASE)  # rx_2_251001_014845
CSV_PAT = re.compile(r".*\.csv$", re.IGNORECASE)


# ======== Tiện ích nhận diện cột ========
def infer_session_from_filename(path: str) -> str:
    bn = os.path.basename(path)
    m = SESSION_RX_RE.search(bn)
    if m:
        yymmdd, hhmmss = m.group(1), m.group(2)
        return f"{yymmdd}_{hhmmss}"
    parent = os.path.basename(os.path.dirname(path))
    return f"SES_{parent}"


def infer_freq_from_path(file_path: str) -> Optional[float]:
    m = re.findall(r"\d{4,5}", os.path.basename(file_path))
    for s in m:
        v = int(s)
        if v in FREQ_WHITELIST:
            return float(v)  # MHz

    m2 = re.findall(r"\d{4,5}", file_path)
    for s in m2:
        v = int(s)
        if v in FREQ_WHITELIST:
            return float(v)
    return None


def pick_col(df: pd.DataFrame, *cands, required=True) -> Optional[str]:
    def norm(s: str) -> str:
        return re.sub(r"[^a-z0-9]", "", s.lower())

    cols = {norm(c): c for c in df.columns}
    for c in cands:
        key = norm(c)
        if key in cols:
            return cols[key]
    if required:
        raise KeyError(f"Missing required column; tried: {cands}. Available: {list(df.columns)}")
    return None


def complex_from_df(df: pd.DataFrame) -> np.ndarray:
    # Ưu tiên Real/Imag, nếu không có thì dùng Mag/Phase (rad)
    has_re = any(c.lower() in {"real", "re", "csi_real", "realpart"} for c in df.columns)
    has_im = any(c.lower() in {"imag", "im", "csi_imag", "imagpart"} for c in df.columns)

    if has_re and has_im:
        col_re = pick_col(df, "CSI_Real", "Real", "Re", "RealPart")
        col_im = pick_col(df, "CSI_Imag", "Imag", "Im", "ImagPart")
        return df[col_re].to_numpy(dtype=np.float64) + 1j * df[col_im].to_numpy(dtype=np.float64)

    col_mag = pick_col(df, "Mag", "Magnitude", "Abs")
    col_ph = pick_col(df, "Phase", "Ph", "Angle", "Arg")
    mag = df[col_mag].to_numpy(dtype=np.float64)
    ph = df[col_ph].to_numpy(dtype=np.float64)
    return mag * np.exp(1j * ph)


# ======== Map chỉ số file -> (row, col) ========
def scan_index_to_grid(idx: int) -> Tuple[int, int]:
    """1..60 -> (row, col)"""
    if not (1 <= idx <= SCAN_NROWS * SCAN_NCOLS):
        raise ValueError(f"scan index out of range: {idx}")
    row = (idx - 1) // SCAN_NCOLS + 1
    col = (idx - 1) % SCAN_NCOLS + 1
    return row, col


def z_index_to_grid(idx: int) -> Tuple[int, int]:
    """1..8 -> (row, col) cho Z ref"""
    if not (1 <= idx <= Z_ROWS * Z_COLS):
        raise ValueError(f"Z ref index out of range: {idx}")
    row = (idx - 1) // Z_COLS + 1
    col = (idx - 1) % Z_COLS + 1
    return row, col


# ======== Dataclass gói dữ liệu ========
@dataclass
class Packet:
    array_tag: str           # "scan0" | "scan1" | "ref" | "scan_op"
    point_id: str            # ví dụ "12" hoặc "Z-3"
    row: int
    col: int
    frame_index: int
    carrier_freq: float      # MHz
    sub_idx: np.ndarray      # (K,)
    csi: np.ndarray          # (K,) complex
    meta: Dict[str, str]     # các trường phụ


# ======== Đọc 1 file CSV rx...csv thành nhiều Packet (gộp theo FrameIndex) ========
def get_carrier_freq_mhz(df: pd.DataFrame) -> Tuple[Optional[np.ndarray], Optional[str]]:
    cand_mhz = ["CenterFreq_MHz", "CenterFrequency_MHz"]
    cand_hz  = ["CarrierFreq_Hz", "CarrierFrequency_Hz", "CenterFrequency_Hz"]
    cand_any = ["CarrierFreq", "CarrierFrequency", "CenterFreq", "CenterFrequency",
                "Freq", "Frequency", "f0", "rf_freq_mhz", "rf_freq_hz"]

    col = pick_col(df, *cand_mhz, required=False)
    if col is not None:
        return df[col].astype(float).to_numpy(), col

    col = pick_col(df, *cand_hz, required=False)
    if col is not None:
        v = df[col].astype(float).to_numpy() / 1e6
        return v, col

    col = pick_col(df, *cand_any, required=False)
    if col is not None:
        v = df[col].astype(float).to_numpy()
        if np.nanmax(np.abs(v)) > 1e6:
            v = v / 1e6
        return v, col

    return None, None


def load_rx_csv_to_packets(file_path: str, array_tag: str, point_id: str, row: int, col: int) -> List[Packet]:
    packets: List[Packet] = []
    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"[skip read] {file_path}: {e}")
        return packets

    try:
        col_fi = pick_col(df, "FrameIndex", "frame_index", "frame index", "fi")
        col_sc = pick_col(df, "SubcarrierIndex", "subcarrier_index", "subcarrier", "sc_idx", "subcarrier index")
    except KeyError as e:
        print(f"[skip cols] {file_path}: {e}")
        return packets

    try:
        csi_complex = complex_from_df(df)
    except KeyError as e:
        print(f"[skip csi] {file_path}: {e}")
        return packets

    work = pd.DataFrame({
        "FrameIndex": df[col_fi].astype(int),
        "SubcarrierIndex": df[col_sc].astype(int),
        "CSI": csi_complex
    })

    cf_mhz, used_col = get_carrier_freq_mhz(df)
    if cf_mhz is not None:
        work["CarrierFreq"] = cf_mhz
    else:
        cf_infer = infer_freq_from_path(file_path)
        if cf_infer is not None:
            work["CarrierFreq"] = cf_infer

    col_cbw  = pick_col(df, "CBW_MHz", required=False)
    col_scbw = pick_col(df, "SubcarrierBandwidth_Hz", required=False)
    col_nt   = pick_col(df, "NumTones", required=False)
    col_ts   = pick_col(df, "Timestamp", required=False)
    col_sys  = pick_col(df, "SystemNS", required=False)
    col_tsf  = pick_col(df, "MPDU_TSF_us", required=False)

    def fft_class_from_idx(sub_idx):
        m = int(np.max(np.abs(sub_idx)))
        if m <= 28:
            return "20MHz-64"
        if m <= 64:
            return "40MHz-128"
        if m <= 128:
            return "80MHz-256"
        return f"unknown({m})"

    meta_cols = [c for c in df.columns if c not in {col_fi, col_sc, used_col}]
    meta_cols = [
        c for c in meta_cols
        if c.lower() not in {
            "mag", "magnitude", "abs", "phase", "ph", "angle", "arg",
            "real", "re", "realpart", "imag", "im", "imagpart",
            "csi_real", "csi_imag"
        }
    ]

    group_keys = ["FrameIndex", "CarrierFreq"] if "CarrierFreq" in work.columns else ["FrameIndex"]
    session_id = infer_session_from_filename(file_path)

    for keys, g in work.groupby(group_keys, sort=True):
        if isinstance(keys, tuple):
            if len(keys) == 2:
                fi, cf = keys
            else:
                fi = keys[0]
                cf = float("nan")
        else:
            fi = keys
            cf = float("nan")

        g = g.sort_values("SubcarrierIndex")
        sub_idx = g["SubcarrierIndex"].to_numpy(dtype=np.int32)
        csi = g["CSI"].to_numpy(dtype=np.complex128)

        meta = {}
        if len(g.index) > 0:
            src_i = g.index[0]
            for m in meta_cols:
                val = df.loc[src_i, m]
                meta[m] = str(val)

        meta["session"] = session_id
        meta["src_path"] = file_path
        meta["phase_linked"] = "0"

        if col_cbw:
            meta["CBW_MHz"] = str(df.loc[src_i, col_cbw])
        if col_scbw:
            meta["SubcarrierBandwidth_Hz"] = str(df.loc[src_i, col_scbw])

        meta["fft_class"] = fft_class_from_idx(sub_idx)

        if col_ts:
            meta["Timestamp"] = str(df.loc[src_i, col_ts])
        if col_sys:
            meta["SystemNS"] = str(df.loc[src_i, col_sys])
        if col_tsf:
            meta["MPDU_TSF_us"] = str(df.loc[src_i, col_tsf])

        if col_nt:
            try:
                nt = int(df.loc[src_i, col_nt])
                if nt != len(np.unique(sub_idx)):
                    meta["warn_NumTones_mismatch"] = f"{nt} vs {len(np.unique(sub_idx))}"
            except Exception:
                pass

        packets.append(Packet(
            array_tag=array_tag,
            point_id=point_id,
            row=row,
            col=col,
            frame_index=int(np.asarray(fi).item()),
            carrier_freq=float(cf),
            sub_idx=sub_idx,
            csi=csi,
            meta=meta
        ))

    return packets


# ======== Duyệt thư mục ========
def _nearest_numeric_ancestor(root: str, file_path: str) -> Optional[str]:
    """
    Lấy numeric ancestor gần file nhất, tính từ thư mục chứa file đi ngược lên đến root.
    Đây là sửa chính để tránh việc opposite data bị collapse về top-level dir.
    """
    root_norm = os.path.normpath(root)
    dir_norm = os.path.normpath(os.path.dirname(file_path))

    try:
        rel = os.path.relpath(dir_norm, root_norm)
    except ValueError:
        rel = dir_norm

    parts = [] if rel in (".", "") else rel.split(os.sep)

    for p in reversed(parts):
        if p.isdigit():
            return str(int(p))
    return None


def _infer_point_id_from_file(root: str, file_path: str) -> str:
    """
    Thứ tự ưu tiên:
    1) numeric ancestor gần file nhất
    2) số trong stem filename
    3) tên stem filename
    """
    pid = _nearest_numeric_ancestor(root, file_path)
    if pid is not None:
        return pid

    stem = os.path.splitext(os.path.basename(file_path))[0]
    m = re.search(r"(\d+)", stem)
    if m:
        return str(int(m.group(1)))

    return stem


def list_point_entries(root: str) -> List[Tuple[str, str]]:
    """
    Trả về list[(point_id, csv_path)].

    KHÁC BẢN CŨ:
    - Không cố định pid theo top-level numeric dir.
    - Mỗi file CSV tự suy point_id từ numeric ancestor gần nó nhất.
    - Giúp xử lý đúng root_opposite có cấu trúc nhiều tầng.
    """
    if root is None or (not os.path.exists(root)):
        return []

    entries: List[Tuple[str, str]] = []

    for dirpath, _, filenames in os.walk(root):
        for fn in sorted(filenames):
            if not CSV_PAT.match(fn):
                continue
            fpath = os.path.join(dirpath, fn)
            pid = _infer_point_id_from_file(root, fpath)
            entries.append((pid, fpath))

    entries.sort(key=lambda x: (str(x[0]), str(x[1])))
    return entries


# ======== Map point_id -> (row,col) ========
def map_scan_rowcol(point_id: str) -> Tuple[int, int]:
    idx = int(point_id)
    return scan_index_to_grid(idx)


def map_z_rowcol(point_id: str) -> Tuple[int, int]:
    idx = int(point_id)
    return z_index_to_grid(idx)


# ======== Dataset ========
@dataclass
class Dataset:
    packets: List[Packet]
    layout: Optional[pd.DataFrame] = None          # giữ tương thích ngược
    layout_main: Optional[pd.DataFrame] = None
    layout_opposite: Optional[pd.DataFrame] = None


def tag_packets_view(
    packets: List[Packet],
    view_name: str,
    layout_name: str,
    geom_role: str
) -> List[Packet]:
    for p in packets:
        p.meta["view"] = view_name           # "main" | "opposite"
        p.meta["layout_name"] = layout_name  # "main" | "opposite"
        p.meta["geom_role"] = geom_role      # "scan" | "ref" | "rx_op"
    return packets


def load_layout(layout_csv: Optional[str], view_name: str = "main") -> Optional[pd.DataFrame]:
    if layout_csv is None:
        return None
    if not os.path.exists(layout_csv):
        raise FileNotFoundError(f"layout file not found: {layout_csv}")

    df = pd.read_csv(layout_csv)

    ren = {}
    for c in df.columns:
        lc = c.lower()
        if lc in {"x", "x_m"}:
            ren[c] = "x"
        elif lc in {"y", "y_m"}:
            ren[c] = "y"
        elif lc in {"z", "z_m"}:
            ren[c] = "z"
        elif lc in {"id", "point_id", "name", "label", "node_id"}:
            ren[c] = "id"
        elif lc in {"node_type", "role", "category", "type"}:
            ren[c] = "type"

    df = df.rename(columns=ren).copy()

    if "id" in df.columns:
        df["id"] = df["id"].astype(str)
    if "type" in df.columns:
        df["type"] = df["type"].astype(str)
    for c in ("x", "y", "z"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    df["_layout_view"] = view_name
    df["_layout_source"] = layout_csv
    return df


def load_dataset(
    root_scan0: str = r"D:/Project/WifiSenssing/Data/Data26_11/Data(0)26_11Csv",
    root_scan1: str = r"D:/Project/WifiSenssing/Data/Data26_11/Data(1)26_11Csv",
    root_zref: str = r"D:/Project/WifiSenssing/Data/Data26_11/ZData26_11Csv",
    layout_csv: Optional[str] = r"D:/Project/WifiSenssing/Code/ForJounal/layout_positions.csv",

    root_opposite: Optional[str] = r"D:/Project/WifiSenssing/Data/Data26_11/OppositePointDataCsv",
    layout_csv_opposite: Optional[str] = r"D:/Project/WifiSenssing/Code/ForJounal/layout_positions_txop_rxop.csv",
    load_opposite: bool = True
) -> Dataset:
    packets: List[Packet] = []

    # ===== Main / old view =====
    for pid, path in list_point_entries(root_scan0):
        try:
            r, c = map_scan_rowcol(pid)
        except Exception as e:
            print(f"[skip scan0 point] pid={pid} path={path} err={e}")
            continue
        pkts = load_rx_csv_to_packets(path, "scan0", pid, r, c)
        packets += tag_packets_view(pkts, view_name="main", layout_name="main", geom_role="scan")

    for pid, path in list_point_entries(root_scan1):
        try:
            r, c = map_scan_rowcol(pid)
        except Exception as e:
            print(f"[skip scan1 point] pid={pid} path={path} err={e}")
            continue
        pkts = load_rx_csv_to_packets(path, "scan1", pid, r, c)
        packets += tag_packets_view(pkts, view_name="main", layout_name="main", geom_role="scan")

    for pid, path in list_point_entries(root_zref):
        try:
            r, c = map_z_rowcol(pid)
        except Exception as e:
            print(f"[skip ref point] pid={pid} path={path} err={e}")
            continue
        pkts = load_rx_csv_to_packets(path, "ref", f"Z-{pid}", r, c)
        packets += tag_packets_view(pkts, view_name="main", layout_name="main", geom_role="ref")

    # ===== Opposite view =====
    if load_opposite and root_opposite is not None:
        if os.path.exists(root_opposite):
            for pid, path in list_point_entries(root_opposite):
                try:
                    r, c = map_scan_rowcol(pid)
                except Exception as e:
                    print(f"[skip scan_op point] pid={pid} path={path} err={e}")
                    continue

                pkts = load_rx_csv_to_packets(path, "scan_op", pid, r, c)
                packets += tag_packets_view(
                    pkts,
                    view_name="opposite",
                    layout_name="opposite",
                    geom_role="rx_op"
                )
        else:
            print(f"[warn] root_opposite not found: {root_opposite}")

    layout_main = load_layout(layout_csv, view_name="main")
    layout_opposite = load_layout(layout_csv_opposite, view_name="opposite") if load_opposite else None

    return Dataset(
        packets=packets,
        layout=layout_main,            # giữ tương thích với code cũ
        layout_main=layout_main,
        layout_opposite=layout_opposite
    )


# ======== Tóm tắt dataset ========
def summarize(ds: Dataset) -> pd.DataFrame:
    rows = []
    for p in ds.packets:
        rows.append({
            "view": p.meta.get("view", ""),
            "layout_name": p.meta.get("layout_name", ""),
            "geom_role": p.meta.get("geom_role", ""),
            "array": p.array_tag,
            "point": p.point_id,
            "row": p.row,
            "col": p.col,
            "f0": p.carrier_freq,
            "frame": p.frame_index,
            "K": len(p.sub_idx)
        })

    df = pd.DataFrame(rows)
    if df.empty:
        return df

    df = df.sort_values(["view", "array", "point", "f0", "frame"])
    return df


# ======== Kiểm tra nhanh ========
if __name__ == "__main__":
    ds = load_dataset(
        root_scan0=r"D:/Project/WifiSenssing/Data/Data26_11/Data(0)26_11Csv",
        root_scan1=r"D:/Project/WifiSenssing/Data/Data26_11/Data(1)26_11Csv",
        root_zref=r"D:/Project/WifiSenssing/Data/Data26_11/ZData26_11Csv",
        layout_csv=r"D:/Project/WifiSenssing/Data/layout_positions.csv",
        root_opposite=r"D:/Project/WifiSenssing/Data/Data26_11/OppositePointDataCsv",
        layout_csv_opposite=r"D:/Project/WifiSenssing/Data/layout_positions_txop_rxop.csv",
        load_opposite=True
    )

    sm = summarize(ds)
    pd.set_option("display.max_rows", 200)

    print(sm.head(50))
    print("\nTổng số packet:", len(ds.packets))
    print("Main layout rows:", 0 if ds.layout_main is None else len(ds.layout_main))
    print("Opposite layout rows:", 0 if ds.layout_opposite is None else len(ds.layout_opposite))

    if not sm.empty:
        print("\nPacket count by view / array:")
        print(sm.groupby(["view", "array"]).size())

        print("\nUnique point count by view / array:")
        print(sm.groupby(["view", "array"])["point"].nunique())

        view layout_name geom_role array point  row  col      f0  frame   K
104281  main        main       ref   ref   Z-1    1    1  5180.0      0  53
104282  main        main       ref   ref   Z-1    1    1  5180.0      1  53
104283  main        main       ref   ref   Z-1    1    1  5180.0      2  53
104284  main        main       ref   ref   Z-1    1    1  5180.0      3  53
104285  main        main       ref   ref   Z-1    1    1  5180.0      4  53
104286  main        main       ref   ref   Z-1    1    1  5180.0      5  53
104287  main        main       ref   ref   Z-1    1    1  5180.0      6  53
104288  main        main       ref   ref   Z-1    1    1  5180.0      7  53
104289  main        main       ref   ref   Z-1    1    1  5180.0      8  53
104290  main        main       ref   ref   Z-1    1    1  5180.0      9  53
104291  main        main       ref   ref   Z-1    1    1  5180.0     10  53
104292  main        main       ref   ref   Z-1    1    1  5180.0     11  53
104293  main

In [2]:
import numpy as np
import pandas as pd
from collections import Counter, defaultdict

ds = load_dataset(load_opposite=True)
sm = summarize(ds)

print("="*80)
print("BASIC SUMMARY")
print("="*80)
print("num_packets:", len(ds.packets))
print("layout_main rows:", 0 if ds.layout_main is None else len(ds.layout_main))
print("layout_opposite rows:", 0 if ds.layout_opposite is None else len(ds.layout_opposite))

print("\nPacket count by view/array/freq:")
print(sm.groupby(["view", "array", "f0"]).size())

print("\nUnique points by view/array:")
print(sm.groupby(["view", "array"])["point"].nunique())

print("\nFrame count per array/point/freq summary:")
g = sm.groupby(["view", "array", "point", "f0"])["frame"].nunique().reset_index(name="n_frames")
print(g.groupby(["view", "array", "f0"])["n_frames"].describe())

print("\nK distribution:")
print(sm.groupby(["view", "array", "f0"])["K"].describe())

print("\nSubcarrier pattern audit:")
patterns = defaultdict(int)
examples = {}
for p in ds.packets:
    key = (
        p.meta.get("view", ""),
        p.array_tag,
        float(p.carrier_freq),
        tuple(p.sub_idx.tolist())
    )
    patterns[key] += 1
    examples[key] = p.sub_idx

rows = []
for key, n in patterns.items():
    view, arr, f0, sub_tuple = key
    rows.append({
        "view": view,
        "array": arr,
        "f0": f0,
        "n_packets": n,
        "K": len(sub_tuple),
        "sub_min": min(sub_tuple),
        "sub_max": max(sub_tuple),
        "sub_hash": hash(sub_tuple)
    })
sub_audit = pd.DataFrame(rows).sort_values(["view", "array", "f0", "n_packets"])
print(sub_audit)

print("\nMetadata key frequency:")
key_counter = Counter()
for p in ds.packets:
    key_counter.update(p.meta.keys())
print(pd.DataFrame(key_counter.items(), columns=["meta_key", "count"]).sort_values("count", ascending=False))

print("\nTimestamp availability:")
for key in ["Timestamp", "SystemNS", "MPDU_TSF_us", "session"]:
    cnt = sum(key in p.meta and str(p.meta[key]).lower() not in ["nan", "none", ""] for p in ds.packets)
    print(key, cnt, "/", len(ds.packets))

print("\nPhase linked flag:")
print(Counter(p.meta.get("phase_linked", "missing") for p in ds.packets))

print("\nCSI sanity:")
amp_vals = []
phase_vals = []
for p in ds.packets[:min(5000, len(ds.packets))]:
    amp_vals.append(np.median(np.abs(p.csi)))
    phase_vals.append(np.angle(p.csi).std())
print("median amplitude sample:", np.nanmedian(amp_vals))
print("phase std sample median:", np.nanmedian(phase_vals))

BASIC SUMMARY
num_packets: 132293
layout_main rows: 156
layout_opposite rows: 37

Packet count by view/array/freq:
view      array    f0    
main      ref      5180.0      812
                   5200.0       48
                   5220.0       67
                   5240.0      722
                   5280.0      273
          scan0    5180.0    15182
                   5200.0    10197
                   5220.0     7021
                   5240.0    13739
                   5280.0     3415
          scan1    5180.0    17151
                   5200.0    11879
                   5220.0     7185
                   5240.0    14642
                   5280.0     3870
opposite  scan_op  5180.0     5026
                   5200.0     6046
                   5220.0     9833
                   5240.0     4505
                   5280.0      680
dtype: int64

Unique points by view/array:
view      array  
main      ref         8
          scan0      60
          scan1      60
opposite  scan_op     9
Na